# **First Project: Resolving Human Resources Problems**

* Name: **Rizky Putra Reinanda**
* Email: **reinanda1908@gmail.com**
* Dicoding ID: **reinanda_13**

# **1. Preparation**

## **1.1 Library Import**

Import all necessary libraries

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

import pandas as pd

df = pd.read_csv('employee_data.csv')

ModuleNotFoundError: No module named 'pandas'

## **1.2 Data Loading**

Load the data from the [source](https://github.com/dicodingacademy/dicoding_dataset/tree/main/employee)

In [ ]:
import pandas as pd

# Pastikan file 'employee_data.csv' ada di folder yang sama
df = pd.read_csv('employee_data.csv')

# Tampilkan data
df.head()

## **1.3 Data Preparation**

### 1.3.1 Dataset Summary

In [ ]:
df.info()

The data contains demographic details, work-related metrics and attrition flag.

* **EmployeeId** - Employee Identifier
* **Attrition** - Did the employee attrition? (0=no, 1=yes)
* **Age** - Age of the employee
* **BusinessTravel** - Travel commitments for the job
* **DailyRate** - Daily salary
* **Department** - Employee Department
* **DistanceFromHome** - Distance from work to home (in km)
* **Education** - 1-Below College, 2-College, 3-Bachelor, 4-Master, 5-Doctor
* **EducationField** - Field of Education
* **EnvironmentSatisfaction** - 1-Low, 2-Medium, 3-High, 4-Very High
* **Gender** - Employee's gender
* **HourlyRate** - Hourly salary
* **JobInvolvement** - 1-Low, 2-Medium, 3-High, 4-Very High
* **JobLevel** - Level of job (1 to 5)
* **JobRole** - Job Roles
* **JobSatisfaction** - 1-Low, 2-Medium, 3-High, 4-Very High
* **MaritalStatus** - Marital Status
* **MonthlyIncome** - Monthly salary
* **MonthlyRate** - Mounthly rate
* **NumCompaniesWorked** - Number of companies worked at
* **Over18** - Over 18 years of age?
* **OverTime** - Overtime?
* **PercentSalaryHike** - The percentage increase in salary last year
* **PerformanceRating** - 1-Low, 2-Good, 3-Excellent, 4-Outstanding
* **RelationshipSatisfaction** - 1-Low, 2-Medium, 3-High, 4-Very High
* **StandardHours** - Standard Hours
* **StockOptionLevel** - Stock Option Level
* **TotalWorkingYears** - Total years worked
* **TrainingTimesLastYear** - Number of training attended last year
* **WorkLifeBalance** - 1-Low, 2-Good, 3-Excellent, 4-Outstanding
* **YearsAtCompany** - Years at Company
* **YearsInCurrentRole** - Years in the current role
* **YearsSinceLastPromotion** - Years since the last promotion
* **YearsWithCurrManager** - Years with the current manager

Check the unique value of each categorical data features

In [ ]:
for feature in df.select_dtypes(include='object'):
    print(feature)
    print(df[feature].unique(), '\n')

Dataset descriptive statistics for numerical and categorical (object) data

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

Based on the analysis of descriptive statistics, all employees are over 18 years old which can be seen in the unique `Over18` feature only 1 data, `Y`. In addition, the `EmployeeCount` feature also only has 1 unique value. So we can eliminate these 2 features because they do not have a major influence on business problems

In [ ]:
df = df.drop(['Over18', 'EmployeeCount'], axis=1)
df

### 1.3.2 Missing Value and Duplicate Data Handling

Check for missing value

In [ ]:
print(df.isnull().sum())

Displaying data that has missing values

In [ ]:
df[df.isnull().any(axis=1)]

Since the dataset has several missing value in the `Attrition` feature, which is the main data feature based on the business problem to be solved, we need to drop the missing value data.

In [ ]:
df.dropna(inplace=True)
print(df.isnull().sum())

Check the amount of data row after the missing value handling process

In [ ]:
df.shape

Check for data duplicates

In [ ]:
df.duplicated().sum()

Check the `Attrition` data type

In [ ]:
df['Attrition'].dtypes

Change the `Attrition` data feature from float to integer, since we already clean it

In [ ]:
df['Attrition'] = df['Attrition'].astype(int)
df['Attrition'].dtypes

### 1.3.3 Ordinal Decoding

Ordinal Decoding for below features:

| Features | 1 | 2 | 3 | 4 | 5 |
|----------|---|---|---|---|---|
| Education                | Below College | College | Bachelor | Master | Doctor |
| EnvironmentSatisfaction  | Low | Medium | High      | Very High   | - |
| JobInvolvement           | Low | Medium | High      | Very High   | - |
| JobSatisfaction          | Low | Medium | High      | Very High   | - |
| PerformanceRating        | Low | Good   | Excellent | Outstanding | - |
| RelationshipSatisfaction | Low | Medium | High      | Very High   | - |
| WorkLifeBalance          | Low | Good   | Excellent | Outstanding | - |

In [ ]:
def ordinal_decoding(df, feature):
    """
    Convert encoded feature in a DataFrame to corresponding categorical labels

    Parameters
        df (pandas.DataFrame) : The DataFrame with feature(s) to be converted
        feature (str or list of str) : The feature name(s) to convert

    Returns
        pandas.DataFrame : The DataFrame with feature(s) as categorical labels
    """

    if feature == 'Attrition':
        category = {0: 'No', 1: 'Yes'}
        df[feature] = df[feature].map(category)
    elif feature == 'Education':
        category = {1: 'Below College', 2: 'College', 3: 'Bachelor',
                    4: 'Master', 5: 'Doctor'}
        df[feature] = df[feature].map(category)
    elif feature in ['WorkLifeBalance', 'PerformanceRating']:
        category = {1: 'Low', 2: 'Good', 3: 'Excellent', 4: 'Outstanding'}
        df[feature] = df[feature].map(category)
    else:
        category = {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'}
        for f in feature:
            df[f] = df[f].map(category)

    return df

Convert the numerical data to categorical data using the defined function

In [ ]:
df = ordinal_decoding(df, 'Attrition')
df = ordinal_decoding(df, 'Education')
df = ordinal_decoding(df, ['EnvironmentSatisfaction', 'JobInvolvement',
                           'JobSatisfaction', 'RelationshipSatisfaction'])
df = ordinal_decoding(df, ['PerformanceRating', 'WorkLifeBalance'])

df

### 1.3.4 Cleaned Data Export

Export the cleaned dataset

In [ ]:
# Dataset for Business Dashboard
df.to_csv('employee_data_cleaned.csv', index=False)

# **2. Data Understanding**

Check the dataset descriptive statistics summary for numerical and categorical data

In [ ]:
df.describe(include='all')

## **2.1 Heatmap Correlation Matrix**

In [ ]:
plt.figure(figsize=(14, 6))
correlation_matrix = df.corr(numeric_only=True).round(2)
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, vmin=-1, vmax=1,annot=True, cmap='coolwarm', mask=mask)
plt.title('Heatmap Correlation Matrix')
plt.show()

Based on the correlation matrix plot above, we can conclude:
1. `Age` has a fairly strong positive correlation with `TotalWorkingYears`, because as age increases, the total years of work are relatively longer.
2. `Age` has a fairly strong positive correlation with `JobLevel`, because as employees get older, employees will generally occupy a higher position in the company.
3. `Age` has a fairly strong positive correlation with `MonthlyIncome`, because as employees get older, employees will generally get a bigger income.
4. `JobLevel` has a very strong positive correlation with `MonthlyIncome`, because more senior employees tend to earn higher incomes.
5. `JobLevel` has a very strong positive correlation with `TotalWorkingYears`, because more senior employees generally have experience and have worked for a longer period of time.
6. `JobLevel` has a fairly strong positive correlation with `YeasAtCompany`, because more senior employees tend to have worked for many years in the same company.
7. `MonthlyIncome` has a very strong positive correlation with `TotalWorkingYears`, because the higher the income, the longer the total number of years of work.
8. `MonthlyIncome` has a fairly strong positive correlation with `YearsAtCompany`, because the higher the income, generally the longer employees have worked at the same company.
9. `TotalWorkingYears` has a fairly strong positive correlation with `YearsAtCompany`, because the longer the span of years of work, generally the longer employees have worked at the same company.
10. `YearsAtCompany` has a very strong positive correlation with `YearsInCurrentRole`.
11. `YearsAtCompany` has a very strong positive correlation with `YearsWithCurrentManager`.
12. `YearsAtCompany` has a fairly strong positive correlation with `YearsSinceLastPromotion`.
13. `YearsInCurrentRole` has a very strong positive correlation with `YearsWithCurrentManager`.
14. `YearsInCurrentRole` has a fairly strong positive correlation with `YearsSinceLastPromotion`.
15. `YearsSinceLastPromotion` has a fairly strong positive correlation with `YearsWithCurrentManager`.

## **2.2 Univariate Analysis**

In [ ]:
numerical, categorical = [], []

for feature in df.columns:
    if df[feature].dtype != 'object':
        numerical.append(feature)
    else:
        categorical.append(feature)

In [ ]:
# @title ### 2.2.1 Numerical Univariate Analysis
df[numerical].hist(bins=15, figsize=(18, 16))
plt.show()

Based on the graph above, we can see that there are several numeric features that are in the form of right-skewed distribution, such as DistanceFromHome, MonhtlyIncome, NumCompaniesWorked, PercentageSalaryHike, TotalWorkingYears, YearsAtCompany, YearsCurrentRole, YearsSinceLastPromotion, YearsWithCurrManager. On the other hand, the Age and TrainingTimeLastYear features are classified as normal distribution.

In [ ]:
# @title ### 2.2.2 Categorical Univariate Analysis
fig, ax = plt.subplots(3, 5, figsize=(24, 14))
for i, feature in enumerate(categorical):
    # Get the row and column index for each subplot
    row = i // 5
    col = i % 5
    sns.countplot(data=df, x=feature, ax=ax[row, col], hue=feature, palette='Set1')
    ax[row, col].set_title(feature, fontsize=10)
    ax[row, col].set_xlabel('')
    ax[row, col].grid()

    # Give count label each bar
    for bar in ax[row, col].patches:
        ax[row, col].annotate(
            str(int(bar.get_height())),
            (bar.get_x() + bar.get_width() / 2, bar.get_height() - (bar.get_height() * 0.05)),
            ha='center', va='top', fontsize=10, color='white', weight='bold')

    # Rotate the x labels
    for label in ax[row, col].get_xticklabels():
        label.set_rotation(90)

# Hide unused subplots if number of features < 16
for j in range(len(categorical), 15):
    fig.delaxes(ax[j // 5, j % 5])

plt.subplots_adjust(hspace=1.2, wspace=0.3)
plt.show()

***Based on the graph above it can be concluded that:***
- The number of male employees is greater than female employees
- The largest number of employees comes from the Research & Development department.
- Many employees have an educational background as a bachelor.
- Most employees come from the field of life science education
- Most employees have a job role as a sales executive
- The number of employees who do not work overtime is greater than the number of employees who do not work overtime.
- Most employees who do not often travel on business (Travel Rarely)
- Most employees have a married status
- Generally employees have high satisfaction with the work environment, high work relations, are very involved in work and have high work performance
- Most employees have a very good work-life balance

## **2.3 Multivariate Analysis**

In [ ]:
# @title ### 2.3.1 Numerical Multivariate Analysis
def numerical_dis_plot(features, df, segment_feature=None, showfliers=True):
    fig, ax = plt.subplots(2, 2, figsize=(15, 8))
    for i, feature in enumerate(features):
        row = i // 2
        col = i % 2
        if segment_feature:
            sns.boxplot(y=segment_feature, x=feature, data=df, ax=ax[row, col], showfliers=showfliers)
            ax[row, col].set_title(f'{segment_feature} Distribution Plot Based on {feature}')
        else:
            sns.boxplot(x=feature, data=df, ax=ax[row, col], showfliers=showfliers)
            ax[row, col].set_title(f'{feature} Distribution Plot')

        ax[row, col].set_ylabel(None)
        ax[row, col].grid(color='lightgray')

    plt.tight_layout()
    plt.show()

In [ ]:
numerical_dis_plot(
    features=['Age', 'TotalWorkingYears', 'MonthlyIncome', 'MonthlyRate'],
    df=df,
    segment_feature='Education'
)

In [ ]:
numerical_dis_plot(
    features=['Age', 'DistanceFromHome', 'JobLevel', 'MonthlyIncome'],
    df=df,
    segment_feature='JobRole'
)

In [ ]:
numerical_dis_plot(
    features=['TotalWorkingYears', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsWithCurrManager'],
    df=df,
    segment_feature='JobRole'
)

In [ ]:
# @title ### 2.3.2 Categorical Multivariate Analysis
def categorical_dis_plot(features, df, segment_feature=None):
    fig, ax = plt.subplots(len(features), 1, figsize=(16, 18))
    for i, feature in enumerate(features):
        if segment_feature:
            sns.countplot(data=df, x=segment_feature, hue=feature, ax=ax[i])
            ax[i].set_title(f'{segment_feature} Distribution Plot Based on {feature}')
        else:
            sns.countplot(data=df, x=feature, ax=ax[i])
            ax[i].set_title(f'{feature} Distribution Plot')

        for bar in ax[i].patches:
            ax[i].annotate(
                str(int(bar.get_height())),
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha='center', va='bottom', fontsize=10)

        ax[i].set_ylabel(None)
        ax[i].grid(True, color='lightgray')

    plt.tight_layout()
    plt.show()

Distribution of categorical features by Job Role

In [ ]:
categorical_dis_plot(
    features=['Gender', 'Education', 'EducationField', 'MaritalStatus'],
    df=df,
    segment_feature='JobRole'
)

In [ ]:
categorical_dis_plot(
    features=['BusinessTravel', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobSatisfaction'],
    df=df,
    segment_feature='JobRole'
)

In [ ]:
categorical_dis_plot(
    features=['OverTime', 'PerformanceRating', 'RelationshipSatisfaction', 'WorkLifeBalance'],
    df=df,
    segment_feature='JobRole'
)

Distribution of categorical features by Department

In [ ]:
categorical_dis_plot(
    features=['Gender', 'Education', 'EducationField', 'MaritalStatus'],
    df=df,
    segment_feature='Department'
)

In [ ]:
categorical_dis_plot(
    features=['BusinessTravel', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobSatisfaction'],
    df=df,
    segment_feature='Department'
)

In [ ]:
categorical_dis_plot(
    features=['OverTime', 'PerformanceRating', 'RelationshipSatisfaction', 'WorkLifeBalance'],
    df=df,
    segment_feature='Department'
)

### 2.3.3 Comparison of Attrition Features

In [ ]:
ax = df['Attrition'].value_counts().plot(kind='bar')
ax.grid()
ax.set_ylabel('Count')

for bar in ax.patches:
    ax.annotate(str(bar.get_height()),
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha='center', va='bottom', fontsize=10)

Based on the Attrition plot above, the number of employees who did not perform attrition (`0`) was greater than those who perform attrition (`1`), so it can be concluded that the data is not balanced or imbalanced data condition.

In [ ]:
categorical_dis_plot(
    features=['Age', 'JobRole', 'JobInvolvement', 'JobLevel'],
    df=df,
    segment_feature='Attrition'
)

From the Attrition plot above, we can conclude:
1. Based on the age, the number of employees who perform attrition when they are 19 years old, and in their twenties and thirties. While the highest attrition rate is at the age of 31. And employees in their fifties such as 53, 54, 57, 59 actually prefer to stay in their company.
2.  Based on the job role, the number of employees who have the highest attrition is those who have the role of Laboratory technician, while the least is those who have the role of Research Director. In addition, employees who have the role of Sales Representative have a fairly high attrition rate that is almost close to the overall population.
3. Based on the job involvement, employees who have low job involvement have a tendency to perform attrition which is clearly seen to be close to the population as a whole.
4. Based on the job level, the higher the job level, the smaller the tendency for employees to perform attrition.

# **3. Data Preprocessing**

In [ ]:
df.head()

In this dataset, the label features used are attrition, whether the employee left the company (1 for yes), and whether or not the employee stayed and left the company (0 for no).

## **3.1 Feature Selection**

Because there are several features that do not contribute or influence the attrition rate in this dataset, for example EmployeeID, these feature need to be removed so that the Machine Learning model only trains the model from the most influential data feature.

In [ ]:
# Drop the unnecessary feature
df = df.drop('EmployeeId', axis=1)
df

## **3.2 Label Encoding**

In [ ]:
numerical, categorical = [], []

for feature in df.columns:
    if df[feature].dtype != 'object':
        numerical.append(feature)
    else:
        categorical.append(feature)

In [ ]:
# Categorical features that need to be encoded
categorical

In [ ]:
le = LabelEncoder()
df[categorical] = df[categorical].apply(le.fit_transform)

## **3.3 Data Normalization**

In [ ]:
# Numerical features that need to be normalized
numerical

In [ ]:
scaler = MinMaxScaler()
df[numerical] = scaler.fit_transform(df[numerical])

## **3.4 Data Checking**

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(4)

## **3.5 Data Splitting**

In [ ]:
# Split the data into independent variable (X) and dependent variable (y) as a label
X = df.drop(['Attrition'], axis=1)
y = df['Attrition']

In [ ]:
# Split the data into train and test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,  random_state=20241116, stratify=y)

print('X_train :', X_train.shape)
print('y_train :', y_train.shape)
print('X_test  :', X_test.shape)
print('y_test  :', y_test.shape)

# **4. Modelling**

In [ ]:
# Models and metrics initialization
models, accuracy, precision, recall, f1 = {}, {}, {}, {}, {}

models['LR'] = LogisticRegression(class_weight='balanced', max_iter=10000)
models['SVM'] = SVC(class_weight='balanced', max_iter=10000)
models['DT'] = DecisionTreeClassifier(class_weight='balanced')
models['GB'] = GradientBoostingClassifier()
models['RF'] = RandomForestClassifier(class_weight='balanced')
models['KNN'] = KNeighborsClassifier()
models['NB'] = GaussianNB()

for i in models.keys():
    models[i].fit(X_train, y_train)

    pred = models[i].predict(X_test)

    accuracy[i] = accuracy_score(pred, y_test)
    precision[i] = precision_score(pred, y_test)
    recall[i] = recall_score(pred, y_test)
    f1[i] = f1_score(pred, y_test)

In [ ]:
model_metrics = pd.DataFrame({
    'Accuracy': accuracy.values(),
    'Precision': precision.values(),
    'Recall': recall.values(),
    'F1-Score': f1.values()
}, index=models.keys())

model_metrics

Based on the evaluation metrics for each model, the Gradient Boosting Classifier has the highest model accuracy among the others, so this model will be used.

In [ ]:
model_gb = GradientBoostingClassifier().fit(X_train, y_train)
model_gb

In [ ]:
y_train_gb = model_gb.predict(X_train)
y_test_gb = model_gb.predict(X_test)

# **5. Evaluation**

In [ ]:
# @title ## **5.1 Evaluation Report Model**
def evaluation_report(y_train, y_predicted, title='Data'):
    print('Classification Report on ' + title)
    print(classification_report(y_train, y_predicted))

    plt.figure(figsize=(4, 3))
    sns.heatmap(confusion_matrix(y_train, y_predicted), annot=True, fmt='d')
    plt.title('Confusion Matrix on ' + title)
    plt.ylabel('True Label Attrition', fontsize=10)
    plt.xlabel('Predicted Label Attrition', fontsize=10)
    plt.yticks(fontsize=8)
    plt.xticks(fontsize=8)
    plt.show()

In [ ]:
evaluation_report(y_train, y_train_gb, title='Training Data')

In [ ]:
evaluation_report(y_test, y_test_gb, title='Testing Data')

## **5.2 Hyperparameter Tuning**

In [ ]:
# Hyperparameter tuning using Grid Search Cross Validation
param_grid = {
    'loss': ['log_loss', 'exponential'],
    'learning_rate': [0.1, 0.01],
    'n_estimators': [100, 250, 500],
    'subsample': [1.0],
    'min_samples_split': [2, 3, 5],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_depth': [3, 10, 20]

}

clf = GridSearchCV(estimator=model_gb, param_grid=param_grid, cv=3, n_jobs=-1, verbose=3)
clf

In [ ]:
# Fit for the best model params estimators
best_model = clf.fit(X_train, y_train)
best_model.best_estimator_

In [ ]:
# Predict the training and testing data using the best model
y_train_best = best_model.predict(X_train)
y_test_best = best_model.predict(X_test)

In [ ]:
evaluation_report(y_train, y_train_best, title='Training Data Best Model')

In [ ]:
evaluation_report(y_test, y_test_best, title='Testing Data Best Model')

# **6 Model Export and Project Requirements**

In [ ]:
joblib.dump(best_model, 'model_gb.joblib')

In [ ]:
!pip freeze > requirements.txt

In [ ]:
# files.download('model_gb.joblib')
# files.download('requirements.txt')